In [1]:
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
from numpy import array, nan, inf, float32

In [2]:
from dataclasses import dataclass

@dataclass
class RangeBounds:
    min: float
    max: float
    min_abs: float

@dataclass
class BitInformationBounds:
    percentile: list[float]
    keepbits: list[int]
    linquant_absolute: list[float]
    bitround_relative: list[float]

@dataclass
class EnsembleSpreadBounds:
    percentile: list[float]
    absolute: list[float]
    relative: list[float]
    quadratic: list[float]

@dataclass
class MinimumBounds:
    percentile: list[float]
    means: list[float]
    counts: list[int]
    bounds: list[list[float]]

In [3]:
def parse_single(path: Path):
    with Path(path).open() as f:
        for l in f.readlines():
            var, rest = l.split(": ", 1)
            info = eval(rest)

            yield var, info

In [4]:
def parse_pressure(path: Path):
    with Path(path).open() as f:
        ls = iter(f.readlines())
        
        l = None
        
        while True:
            if l is None:
                try:
                    l = next(ls)
                except:
                    break
        
            var, rest = l.split(":", 1)
        
            while True:
                try:
                    l = next(ls)
                except:
                    l = None
                    break
        
                if len(l.strip()) == 0:
                    continue
        
                if not l.startswith(" - "):
                    break
        
                level, rest = l[len(" - "):].split(": ", 1)
                info = eval(rest)
        
                yield var, level, info

In [5]:
table = SimpleNamespace(
    var=[], level=[], percentile=[],
    min=[], range=[], max=[],
    lpbits=[], lpabsolute=[], lprelative=[],
    brlqabsolute=[], brabsolute=[], brrelative=[],
    esabsolute=[], esrelative=[], esquadratic=[],
    unabsolute=[],
    cabsolute=[], crelative=[], cquadratic=[], pick=[],
)

for (
    (rvar, rinfo),
    (lvar, linfo),
    (bvar, binfo),
    (evar, einfo),
    (uvar, uinfo),
    (mvar, minfo),
) in zip(
    parse_single("range-single.txt"),
    parse_single("linear-packing-single.txt"),
    parse_single("bit-information-single.txt"),
    parse_single("ensemble-spread-single.txt"),
    parse_single("uniformity-single.txt"),
    parse_single("minimum-single.txt"),
):
    assert all(v == rvar for v in [rvar, lvar, bvar, evar, uvar, mvar])
    assert binfo.percentile == einfo.percentile and minfo.percentile == einfo.percentile
    
    for p, k, a, r, q, b in zip(
        binfo.percentile, binfo.keepbits,
        einfo.absolute, einfo.relative, einfo.quadratic,
        minfo.bounds,
    ):
        table.var.append(bvar)
        table.level.append("single")
        table.percentile.append(f"{p:.0%}")
        
        table.min.append(f"{rinfo.min:.2e}")
        table.range.append(f"{rinfo.max - rinfo.min:.2e}")
        table.max.append(f"{rinfo.max:.2e}")

        table.lpbits.append(f"{linfo}")
        table.lpabsolute.append(f"{(rinfo.max - rinfo.min) / np.exp2(linfo) * 0.5:.1e}")
        table.lprelative.append(f"{((rinfo.max - rinfo.min) / np.exp2(linfo) * 0.5) / rinfo.max:.2%}")
        
        table.brlqabsolute.append(f"{(rinfo.max - rinfo.min) / np.exp2(k-9) * 0.5:.1e}")
        table.brabsolute.append(f"{np.exp2(int(np.log2(rinfo.min_abs))) / np.exp2(k-9):.1e}")
        table.brrelative.append(f"{np.exp2(-(k-9)) * 0.5:.2%}")
        
        table.esabsolute.append(f"{a:.1e}")
        table.esrelative.append(f"{r:.2%}")
        table.esquadratic.append(f"{q:.1e}")

        table.unabsolute.append(f"{uinfo[0]:.1e}")

        cabs = f"{np.nansum(np.abs(np.array(b) - a) * minfo.counts) / np.sum(minfo.counts):.1e}"
        crel = f"{np.nansum(np.abs(np.array(b) - np.abs(minfo.means) * r) * minfo.counts) / np.sum(minfo.counts):.1e}"
        cqua = f"{np.nansum(np.abs(np.array(b) - (q - q * (-1.0 + 2.0 * (np.array(minfo.means) - rinfo.min) / (rinfo.max - rinfo.min))**2)) * minfo.counts) / np.sum(minfo.counts):.1e}"

        table.cabsolute.append(cabs)
        table.crelative.append(crel)
        table.cquadratic.append(cqua)

        bounds = [("absolute", cabs), ("relative", crel), ("quadratic", cqua)]
        table.pick.append(sorted(bounds, key=lambda x: float(x[1]))[0][0])

for (
    (rvar, rlevel, rinfo),
    (lvar, llevel, linfo),
    (bvar, blevel, binfo),
    (evar, elevel, einfo),
    (uvar, ulevel, uinfo),
    (mvar, mlevel, minfo),
) in zip(
    parse_pressure("range-pressure.txt"),
    parse_pressure("linear-packing-pressure.txt"),
    parse_pressure("bit-information-pressure.txt"),
    parse_pressure("ensemble-spread-pressure.txt"),
    parse_pressure("uniformity-pressure.txt"),
    parse_pressure("minimum-pressure.txt"),
):
    assert all(v == rvar for v in [rvar, lvar, bvar, evar, uvar, mvar])
    assert all(l == rlevel for l in [rlevel, llevel, blevel, elevel, ulevel, mlevel])
    assert binfo.percentile == einfo.percentile and minfo.percentile == einfo.percentile
    
    for p, k, a, r, q, b in zip(
        binfo.percentile, binfo.keepbits,
        einfo.absolute, einfo.relative, einfo.quadratic,
        minfo.bounds,
    ):
        table.var.append(bvar)
        table.level.append(blevel)
        table.percentile.append(f"{p:.0%}")
        
        table.min.append(f"{rinfo.min:.2e}")
        table.range.append(f"{rinfo.max - rinfo.min:.2e}")
        table.max.append(f"{rinfo.max:.2e}")

        table.lpbits.append(f"{linfo}")
        table.lpabsolute.append(f"{(rinfo.max - rinfo.min) / np.exp2(linfo) * 0.5:.1e}")
        table.lprelative.append(f"{((rinfo.max - rinfo.min) / np.exp2(linfo) * 0.5) / rinfo.max:.2%}")
        
        table.brlqabsolute.append(f"{(rinfo.max - rinfo.min) / np.exp2(k-9) * 0.5:.1e}")
        table.brabsolute.append(f"{np.exp2(np.floor(np.log2(rinfo.min_abs))) / np.exp2(k-9):.1e}")
        table.brrelative.append(f"{np.exp2(-(k-9)) * 0.5:.2%}")
        
        table.esabsolute.append(f"{a:.1e}")
        table.esrelative.append(f"{r:.2%}")
        table.esquadratic.append(f"{q:.1e}")

        table.unabsolute.append(f"{uinfo[0]:.1e}")

        cabs = f"{np.nansum(np.abs(np.array(b) - a) * minfo.counts) / np.sum(minfo.counts):.1e}"
        crel = f"{np.nansum(np.abs(np.array(b) - np.abs(minfo.means) * r) * minfo.counts) / np.sum(minfo.counts):.1e}"
        cqua = f"{np.nansum(np.abs(np.array(b) - (q - q * (-1.0 + 2.0 * (np.array(minfo.means) - rinfo.min) / (rinfo.max - rinfo.min))**2)) * minfo.counts) / np.sum(minfo.counts):.1e}"

        table.cabsolute.append(cabs)
        table.crelative.append(crel)
        table.cquadratic.append(cqua)

        bounds = [("absolute", cabs), ("relative", crel), ("quadratic", cqua)]
        table.pick.append(sorted(bounds, key=lambda x: float(x[1]))[0][0])

df_raw = pd.DataFrame.from_dict(table.__dict__).set_index(["var", "level", "percentile"])

/tmp/junityre/26953288/ipykernel_3413410/1682786590.py:101: RuntimeWarning: invalid value encountered in scalar divide
  table.lprelative.append(f"{((rinfo.max - rinfo.min) / np.exp2(linfo) * 0.5) / rinfo.max:.2%}")
/tmp/junityre/26953288/ipykernel_3413410/1682786590.py:104: RuntimeWarning: divide by zero encountered in log2
  table.brabsolute.append(f"{np.exp2(np.floor(np.log2(rinfo.min_abs))) / np.exp2(k-9):.1e}")
/tmp/junityre/26953288/ipykernel_3413410/1682786590.py:115: RuntimeWarning: invalid value encountered in divide
  cqua = f"{np.nansum(np.abs(np.array(b) - (q - q * (-1.0 + 2.0 * (np.array(minfo.means) - rinfo.min) / (rinfo.max - rinfo.min))**2)) * minfo.counts) / np.sum(minfo.counts):.1e}"


In [6]:
df_raw.to_csv("table-raw.csv")

In [7]:
df_display = df_raw.rename(columns=dict(
    min="minimum",
    range="max - min",
    max="maximum",

    lpbits="#bits (lp)",
    lpabsolute="absolute (lp)",
    lprelative="relative (lp)",

    brlqabsolute="absolute (br+lq)",
    brabsolute="absolute (br)",
    brrelative="relative (br)",
    
    esabsolute="absolute (es)",
    esrelative="relative (es)",
    esquadratic="quadratic (es)",

    unabsolute="absolute (un)",

    cabsolute="eb_abs cost",
    crelative="eb_rel cost",
    cquadratic="eb_qua cost",

    pick="error bound",
))

In [8]:
df_display.to_csv("table.csv")

In [9]:
from IPython.display import HTML

HTML(df_display.to_html())